In [1]:
!pip install -q transformers datasets scikit-learn

In [2]:
from datasets import load_dataset

dataset = load_dataset("freococo/burmese-contextual-pragmatics")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['uid', 'root_meaning', 'instruction', 'context', 'style', 'tone', 'utterance', 'phonetic', 'register', 'politeness', 'emotion', 'power_relation', 'intent_strength', 'notes'],
        num_rows: 2200
    })
})


In [3]:
# ── Section 1: Preprocessing ─────────────────────────────────────────────────

from collections import Counter
from sklearn.preprocessing import LabelEncoder
import torch

train_ds = dataset["train"]

# ── 1a. Politeness coarsening (6 classes) ────────────────────────────────────
politeness_map = {
    "neutral": "neutral", "polite": "polite", "informal": "informal",
    "professional": "professional", "blunt": "blunt", "rude": "rude",
    "very_polite": "polite", "friendly": "informal", "religious": "polite",
    "impolite": "rude", "stern": "blunt", "sarcastic": "rude",
    "polite_but_firm": "polite"
}

# ── 1b. Tone coarsening (5 classes) ──────────────────────────────────────────
positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal   = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impatient",
            "Suspicious","Rude","Blunt","Crude","Envious","Dangerous",
            "Threatening","Fed up","Dry","Indifferent"}
emotional= {"Urgent","Pleading","Dramatic","Sad","Distressed","Tired",
            "Exhausted","Panicked","Shocked","Surprised","Concerned",
            "Apologetic","Appologetic","Regretful","Emotional","Teasing",
            "Casual","Dazed","Confused","Alarmed","Overwhelmed","Startled",
            "Devastated","Worried","Nervous/Sincere","Sad/Soft","Sad/Sweet",
            "Distress","Painful","Sorrowful","Weak","Uncertain","Weary",
            "Hurry","Impatient","Relieved","Fearful","Scared","Frightened",
            "Grieved","Incredulous","Bored","Lazy","Sleepy","Stressed",
            "Exasperated","Frustrated","Mischievous","Suggestive","Spooky",
            "Whispering","Indirect","Exclamatory","Lazy/Neutral",
            "Casual/Sleepy","Cheerful/Tired","Sweet/Weary","Sad/Serious",
            "Soft/Vulnerable","Natural","Considerate","Faithful","Trusting"}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal:    return "formal"
    if t in negative:  return "negative"
    if t in emotional: return "emotional"
    return "neutral"

# ── 1c. Power relation — keep as-is but merge tiny classes ───────────────────
power_map = {
    "equal":                "equal",
    "inferior_to_superior": "inferior_to_superior",
    "superior_to_inferior": "superior_to_inferior",
    "any":                  "any",
    "customer_to_staff":    "inferior_to_superior",
    "customer_to_seller":   "inferior_to_superior",
    "passenger_to_driver":  "inferior_to_superior",
    "patient_to_doctor":    "inferior_to_superior",
}

# ── 1d. Register — already clean (4 classes) ─────────────────────────────────

# ── 1e. Apply all mappings ────────────────────────────────────────────────────
def apply_all_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"], "neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"], "equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_all_labels)

# ── 1f. Fit label encoders ────────────────────────────────────────────────────
le_politeness = LabelEncoder().fit(train_ds["politeness_coarse"])
le_tone       = LabelEncoder().fit(train_ds["tone_coarse"])
le_power      = LabelEncoder().fit(train_ds["power_coarse"])
le_register   = LabelEncoder().fit(train_ds["register_label"])

print("Politeness classes:", list(le_politeness.classes_))
print("Tone classes:      ", list(le_tone.classes_))
print("Power classes:     ", list(le_power.classes_))
print("Register classes:  ", list(le_register.classes_))

# ── 1g. Add integer labels ────────────────────────────────────────────────────
def encode_all_labels(example):
    example["label_politeness"] = int(le_politeness.transform([example["politeness_coarse"]])[0])
    example["label_tone"]       = int(le_tone.transform([example["tone_coarse"]])[0])
    example["label_power"]      = int(le_power.transform([example["power_coarse"]])[0])
    example["label_register"]   = int(le_register.transform([example["register_label"]])[0])
    return example

train_ds = train_ds.map(encode_all_labels)

# ── 1h. Train/val/test split ──────────────────────────────────────────────────
split1 = train_ds.train_test_split(test_size=0.30, seed=42)
split2 = split1["test"].train_test_split(test_size=0.50, seed=42)

train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"\nSplit — Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

# ── 1i. Class weights for all four tasks ─────────────────────────────────────
def get_weights(data, label_col, n_classes):
    counts = Counter(data[label_col])
    total  = len(data)
    return torch.tensor(
        [total / (n_classes * counts[i]) for i in range(n_classes)],
        dtype=torch.float
    )

weights_politeness = get_weights(train_data, "label_politeness", len(le_politeness.classes_))
weights_tone       = get_weights(train_data, "label_tone",       len(le_tone.classes_))
weights_power      = get_weights(train_data, "label_power",      len(le_power.classes_))
weights_register   = get_weights(train_data, "label_register",   len(le_register.classes_))

print("\nWeights politeness:", weights_politeness.tolist())
print("Weights tone:      ", weights_tone.tolist())
print("Weights power:     ", weights_power.tolist())
print("Weights register:  ", weights_register.tolist())

Politeness classes: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]
Tone classes:       [np.str_('emotional'), np.str_('formal'), np.str_('negative'), np.str_('neutral'), np.str_('positive')]
Power classes:      [np.str_('any'), np.str_('equal'), np.str_('inferior_to_superior'), np.str_('superior_to_inferior')]
Register classes:   [np.str_('colloquial'), np.str_('formal'), np.str_('slang'), np.str_('standard')]

Split — Train: 1540  Val: 330  Test: 330

Weights politeness: [5.238095283508301, 0.9759188890457153, 0.31844499707221985, 0.8174097537994385, 3.8888888359069824, 6.111111164093018]
Weights tone:       [1.1800765991210938, 1.2571429014205933, 4.219178199768066, 0.4873417615890503, 0.936170220375061]
Weights power:      [3.377192974090576, 0.33018869161605835, 2.566666603088379, 3.5]
Weights register:   [0.3635505139827728, 4.010416507720947, 3.564814805984497, 1.3898917436599731]


In [4]:
# ── Section 2: Tokenizer & Tokenization Functions ────────────────────────────

from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# columns to keep after tokenization
KEEP_COLS = ["input_ids", "attention_mask", "label"]

def tokenize(data, fn, label_col):
    """Helper: apply fn, rename label col, remove extras, set torch format."""
    def wrapped(example):
        enc = fn(example)
        enc["label"] = example[label_col]
        return enc
    cols_to_remove = [c for c in data.column_names if c not in KEEP_COLS]
    out = data.map(wrapped, remove_columns=cols_to_remove)
    out.set_format("torch")
    return out

# ── Tokenization functions (one per classifier) ───────────────────────────────

def tok_register(example):
    # Classifier A — utterance only
    return tokenizer(
        example["utterance"],
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

def tok_power(example):
    # Classifier B — utterance + context
    text = example["utterance"] + " </s> " + str(example["context"]).strip()
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

def tok_tone(example):
    # Classifier C — utterance + context
    text = example["utterance"] + " </s> " + str(example["context"]).strip()
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

def tok_stage1(example):
    # Politeness Stage 1 — utterance only
    return tokenizer(
        example["utterance"],
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

def tok_stage2(example):
    # Politeness Stage 2 — utterance + context + instruction
    text = (
        example["utterance"]
        + " </s> " + str(example["context"]).strip()
        + " </s> " + str(example["instruction"]).strip()
    )
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

def tok_stage3(example):
    # Politeness Stage 3 — full oracle (ground truth metadata)
    prefix = (
        f"[register: {example['register']}] "
        f"[power: {example['power_relation']}] "
        f"[tone: {example['tone_coarse']}] "
    )
    text = (
        prefix + example["utterance"]
        + " </s> " + str(example["context"]).strip()
        + " </s> " + str(example["instruction"]).strip()
    )
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

def tok_stage4(example):
    # Politeness Stage 4 — predicted metadata (filled at inference time)
    # prefix will use predicted register/power/tone from classifiers A, B, C
    prefix = (
        f"[register: {example['pred_register']}] "
        f"[power: {example['pred_power']}] "
        f"[tone: {example['pred_tone']}] "
    )
    text = (
        prefix + example["utterance"]
        + " </s> " + str(example["context"]).strip()
        + " </s> " + str(example["instruction"]).strip()
    )
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

print("Tokenizer loaded:", type(tokenizer).__name__)
print("All tokenization functions defined.")
print("\nReady to tokenize. Functions available:")
print("  tok_register  → Classifier A (register)")
print("  tok_power     → Classifier B (power relation)")
print("  tok_tone      → Classifier C (tone)")
print("  tok_stage1    → Politeness Stage 1")
print("  tok_stage2    → Politeness Stage 2")
print("  tok_stage3    → Politeness Stage 3 (oracle)")
print("  tok_stage4    → Politeness Stage 4 (end-to-end, needs predictions)")

Tokenizer loaded: XLMRobertaTokenizer
All tokenization functions defined.

Ready to tokenize. Functions available:
  tok_register  → Classifier A (register)
  tok_power     → Classifier B (power relation)
  tok_tone      → Classifier C (tone)
  tok_stage1    → Politeness Stage 1
  tok_stage2    → Politeness Stage 2
  tok_stage3    → Politeness Stage 3 (oracle)
  tok_stage4    → Politeness Stage 4 (end-to-end, needs predictions)


In [5]:
# ── Section 3: Trainer Setup & HF Login ──────────────────────────────────────

import torch
import torch.nn as nn
import numpy as np
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import login

# ── Weighted Trainer ──────────────────────────────────────────────────────────
# We subclass Trainer to inject weighted cross-entropy loss.
# class_weights is set before each training run (different per classifier).

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss    = nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy":    round(accuracy_score(labels, preds), 4),
        "macro_f1":    round(f1_score(labels, preds, average="macro"), 4),
        "weighted_f1": round(f1_score(labels, preds, average="weighted"), 4),
    }

# ── Training args factory ─────────────────────────────────────────────────────
# Returns a fresh TrainingArguments for each classifier.
def make_training_args(hub_repo, batch_size=16):
    return TrainingArguments(
        output_dir=hub_repo,
        num_train_epochs=8,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=2e-5,
        warmup_steps=150,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=50,
        report_to="none",
        seed=42,
        push_to_hub=True,
        hub_model_id=hub_repo,
        hub_strategy="every_save"
    )

# ── HF Login ──────────────────────────────────────────────────────────────────
login()

print("\nSetup complete. Ready to train.")
print("WeightedTrainer, compute_metrics, make_training_args all defined.")


Setup complete. Ready to train.
WeightedTrainer, compute_metrics, make_training_args all defined.


In [7]:
# ── Section 4: Classifier A — Register ───────────────────────────────────────

import gc
from transformers import AutoModelForSequenceClassification

# Tokenize
print("Tokenizing...")
tok_train_A = tokenize(train_data, tok_register, "label_register")
tok_val_A   = tokenize(val_data,   tok_register, "label_register")
tok_test_A  = tokenize(test_data,  tok_register, "label_register")
print("Done.")

# Model
model_A = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le_register.classes_)
)

# Train
trainer_A = WeightedTrainer(
    class_weights=weights_register,
    model=model_A,
    args=make_training_args("annasus10/xlmr-burmese-register"),
    train_dataset=tok_train_A,
    eval_dataset=tok_val_A,
    compute_metrics=compute_metrics
)

trainer_A.train()
trainer_A.push_to_hub()
print("Classifier A pushed to HuggingFace.")

Tokenizing...


Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Done.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...egister/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...egister/model.safetensors:  14%|#3        |  152MB / 1.11GB            

Classifier A pushed to HuggingFace.


In [8]:
# Classifier A test evaluation
from sklearn.metrics import classification_report
import numpy as np

results_A = trainer_A.evaluate(tok_test_A)
print("\nClassifier A (Register) Test Results:")
for k, v in results_A.items():
    print(f"  {k}: {v}")

preds_A_out = trainer_A.predict(tok_test_A)
preds_A = np.argmax(preds_A_out.predictions, axis=1)
true_A  = preds_A_out.label_ids

print("\nClassifier A Classification Report:")
print(classification_report(true_A, preds_A, target_names=le_register.classes_))


Classifier A (Register) Test Results:
  eval_loss: 0.6840561628341675
  eval_accuracy: 0.7394
  eval_macro_f1: 0.7045
  eval_weighted_f1: 0.7521
  eval_runtime: 2.1675
  eval_samples_per_second: 152.247
  eval_steps_per_second: 5.075
  epoch: 8.0

Classifier A Classification Report:
              precision    recall  f1-score   support

  colloquial       0.90      0.75      0.82       222
      formal       0.66      1.00      0.79        21
       slang       0.75      0.62      0.68        29
    standard       0.44      0.67      0.53        58

    accuracy                           0.74       330
   macro avg       0.69      0.76      0.70       330
weighted avg       0.79      0.74      0.75       330



In [9]:
# ── Section 5: Classifier B — Power Relation ─────────────────────────────────

import gc, torch

# Clear Classifier A from GPU
del model_A
gc.collect()
torch.cuda.empty_cache()

# Tokenize
print("Tokenizing...")
tok_train_B = tokenize(train_data, tok_power, "label_power")
tok_val_B   = tokenize(val_data,   tok_power, "label_power")
tok_test_B  = tokenize(test_data,  tok_power, "label_power")
print("Done.")

# Model
model_B = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le_power.classes_)
)

# Train
trainer_B = WeightedTrainer(
    class_weights=weights_power,
    model=model_B,
    args=make_training_args("annasus10/xlmr-burmese-power"),
    train_dataset=tok_train_B,
    eval_dataset=tok_val_B,
    compute_metrics=compute_metrics
)

trainer_B.train()
trainer_B.push_to_hub()
print("Classifier B pushed to HuggingFace.")

Tokenizing...


Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Done.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e-power/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...e-power/model.safetensors:   9%|9         |  101MB / 1.11GB            

Classifier B pushed to HuggingFace.


In [10]:
# Classifier B test evaluation
results_B = trainer_B.evaluate(tok_test_B)
print("\nClassifier B (Power Relation) Test Results:")
for k, v in results_B.items():
    print(f"  {k}: {v}")

preds_B_out = trainer_B.predict(tok_test_B)
preds_B = np.argmax(preds_B_out.predictions, axis=1)
true_B  = preds_B_out.label_ids

print("\nClassifier B Classification Report:")
print(classification_report(true_B, preds_B, target_names=le_power.classes_))


Classifier B (Power Relation) Test Results:
  eval_loss: 1.1271076202392578
  eval_accuracy: 0.7394
  eval_macro_f1: 0.5983
  eval_weighted_f1: 0.7546
  eval_runtime: 2.1569
  eval_samples_per_second: 152.994
  eval_steps_per_second: 5.1
  epoch: 8.0

Classifier B Classification Report:
                      precision    recall  f1-score   support

                 any       0.35      0.47      0.40        30
               equal       0.89      0.79      0.84       241
inferior_to_superior       0.69      0.76      0.72        33
superior_to_inferior       0.36      0.54      0.43        26

            accuracy                           0.74       330
           macro avg       0.57      0.64      0.60       330
        weighted avg       0.78      0.74      0.75       330



In [11]:
import shutil, gc, torch

# Delete all local checkpoint folders
shutil.rmtree("./annasus10", ignore_errors=True)
gc.collect()
torch.cuda.empty_cache()

import shutil as sh
total, used, free = sh.disk_usage("/")
print(f"Free: {free/1e9:.1f} GB")

Free: 73.6 GB


In [12]:
# ── Section 6: Classifier C — Tone ───────────────────────────────────────────

import gc, torch
from transformers import AutoModelForSequenceClassification

# Clear Classifier B
del model_B
gc.collect()
torch.cuda.empty_cache()

# Tokenize
print("Tokenizing...")
tok_train_C = tokenize(train_data, tok_tone, "label_tone")
tok_val_C   = tokenize(val_data,   tok_tone, "label_tone")
tok_test_C  = tokenize(test_data,  tok_tone, "label_tone")
print("Done.")

# Model
model_C = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le_tone.classes_)
)

# Training args with save_total_limit=1
from transformers import TrainingArguments

args_C = TrainingArguments(
    output_dir="annasus10/xlmr-burmese-tone",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42,
    push_to_hub=True,
    hub_model_id="annasus10/xlmr-burmese-tone",
    hub_strategy="every_save"
)

trainer_C = WeightedTrainer(
    class_weights=weights_tone,
    model=model_C,
    args=args_C,
    train_dataset=tok_train_C,
    eval_dataset=tok_val_C,
    compute_metrics=compute_metrics
)

trainer_C.train()
trainer_C.push_to_hub()
print("Classifier C pushed to HuggingFace.")

Tokenizing...


Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Done.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...se-tone/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...se-tone/model.safetensors:  12%|#1        |  130MB / 1.11GB            

Classifier C pushed to HuggingFace.


In [13]:
results_C = trainer_C.evaluate(tok_test_C)
print("\nClassifier C (Tone) Test Results:")
for k, v in results_C.items():
    print(f"  {k}: {v}")

preds_C_out = trainer_C.predict(tok_test_C)
preds_C = np.argmax(preds_C_out.predictions, axis=1)
true_C  = preds_C_out.label_ids

print("\nClassifier C Classification Report:")
print(classification_report(true_C, preds_C, target_names=le_tone.classes_))


Classifier C (Tone) Test Results:
  eval_loss: 1.1129928827285767
  eval_accuracy: 0.6667
  eval_macro_f1: 0.6128
  eval_weighted_f1: 0.6713
  eval_runtime: 2.1745
  eval_samples_per_second: 151.759
  eval_steps_per_second: 5.059
  epoch: 8.0

Classifier C Classification Report:
              precision    recall  f1-score   support

   emotional       0.53      0.49      0.51        55
      formal       0.68      0.71      0.69        58
    negative       0.33      0.53      0.41        17
     neutral       0.82      0.68      0.74       132
    positive       0.65      0.78      0.71        68

    accuracy                           0.67       330
   macro avg       0.60      0.64      0.61       330
weighted avg       0.69      0.67      0.67       330



In [14]:
import shutil, gc, torch

# Delete all local checkpoint folders
shutil.rmtree("./annasus10", ignore_errors=True)
gc.collect()
torch.cuda.empty_cache()

import shutil as sh
total, used, free = sh.disk_usage("/")
print(f"Free: {free/1e9:.1f} GB")

Free: 73.6 GB


In [6]:
import gc, torch

# Delete all tokenized datasets
for var in ['tok_train_S1', 'tok_val_S1', 'tok_test_S1',
            'tok_train_S2', 'tok_val_S2', 'tok_test_S2',
            'tok_train_S3', 'tok_val_S3', 'tok_test_S3',
            'trainer_C', 'model_S1']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

GPU free: 15.53 GB / 15.64 GB
